# 09 — Geographic encoding ablation on the fixed A2 30k baseline

**Prespecify the rules before running; do not revise them in response to results.** Keep the frozen EMSCAD train/test split and grouping policy. This notebook reads only `d1_train.csv`, never the test set. Restore notebook 02's exact 5 repeats × 5 outer folds, with seed 42. Use three grouped inner folds and C=[0.1,0.3,1.0], selected by mean inner PR-AUC. Select the F1 cutoff only from inner out-of-fold predictions. Preserve the existing country one-hot features and `country_us` unchanged.

|Variant|Only change from A2|
|---|---|
|V0|Refit balanced A2 logistic regression with 30k TF-IDF; reproduce historical PR-AUC≈0.605|
|V1|Append state log1p(count), unseen, and a missing flag to distinguish MISSING|
|V2|Append the same state and city features|
|V3|As V2, but each training group contributes at most once to each location, at every hierarchy level|

Split location at its first two commas into country/state/city. Uppercase country, casefold state/city, and normalise whitespace. Do not infer locations or merge aliases. Keys include their parents: country|state and country|state|city. For training counts below 5, back off city→state→country→UNSEEN. Missing keys, including missing parents, remain MISSING without borrowing parent counts. Represent MISSING as count=0, missing=1, and unsupported UNSEEN as count=0, missing=0. Determine unseen before backoff: only an originally non-missing key with zero training support has unseen=1; rare but observed keys have unseen=0. Country counts support backoff only; no separate country-count column is appended.

The encoder and its StandardScaler live inside the pipeline and are fitted separately in every inner/outer training split. Validation only transforms. V3 uses group_id for deduplication, never as a model feature. Training rows may contribute to unsupervised training counts; validation rows never do.

**Decision rules:** a variant must have a Nadeau–Bengio corrected 95% interval entirely above zero versus V0. Intervals containing zero or entirely below zero are negative results and are excluded. If only V2 passes and V3 does not, label V2 as primarily reflecting repeated postings and exclude it; this is a diagnostic classification, not a verified causal explanation. Use V1−V0 and V2−V1 to inspect granularity, and V3−V2 to inspect deduplication. Do not change folds, cutoff rules, or other features after seeing results. Intervals are exploratory and not adjusted for multiple comparisons.

**Metric definitions:** primary PR-AUC is sklearn average precision, reported as outer-fold mean and sample SD. The corrected interval is mean difference ± t(24)×sqrt[(1/25+mean n_val/n_train)×sample variance of paired differences]. Precision@Recall60 is the maximum outer-curve precision with recall≥0.60; it is descriptive and does not select a deployment cutoff. Top2%/5% uses floor(budget×outer validation rows), with stable row-order tie breaking. Decisions use unrounded values.

**Running:** Run All evaluates V0–V3 and saves resumable outer-fold checkpoints. Set RUN_VARIANTS=[] to read cached results only. V0 must reproduce historical mean PR-AUC within 0.005; otherwise stop and investigate the configuration/environment instead of changing folds to recover a score. V0 is refitted, not substituted with historical results. Reports: `reports/geography_ablation`; row predictions and contributions: `data/processed/geography_ablation`. Interrupt the kernel before closing if necessary: completed folds can be reused, while an interrupted fold restarts.

**Unseen-column limitation:** every training row's non-missing key appears at least once, so its unseen flag is always zero and logistic regression cannot learn a nonzero coefficient for that column. Retain it for validation diagnostics; do not manufacture training variation from validation data or introduce unplanned cross-fitting. Backoff counts and missing flags remain learnable.


In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import sys
import tempfile
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Memory
from sklearn.base import clone
from threadpoolctl import threadpool_limits

ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/features.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.features import build_inputs
from src.evaluation import validate_folds,fold_signature,select_settings,score_summary
from src.geography import geographic_pipeline,auxiliary_metrics,corrected_difference

RUN_VARIANTS=['V0','V1','V2','V3']
SEED=42
INNER_SPLITS=3
PARAM_GRID={'model__C':[0.1,0.3,1.0]}
THREADS=2
BASELINE_TOLERANCE=0.005
REPORT_DIR=ROOT/'reports/geography_ablation'
PROCESSED_DIR=ROOT/'data/processed/geography_ablation'
NAMES={'V0':'A2 30k baseline','V1':'state row counts','V2':'state+city row counts','V3':'state+city group-location counts'}
RESULTS={}

def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def digest(value):return hashlib.sha256(json.dumps(value,sort_keys=True,allow_nan=False).encode()).hexdigest()
def write_json(path,value):
    path=Path(path);temp=path.with_name(path.name+'.tmp')
    temp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False));temp.replace(path)
def write_csv(path,frame):
    path=Path(path);temp=path.with_name(path.name+'.tmp')
    frame.to_csv(temp,index=False);temp.replace(path)

if set(RUN_VARIANTS)-set(NAMES):raise ValueError('Unknown variant.')
if SEED!=42 or INNER_SPLITS!=3 or PARAM_GRID!={'model__C':[0.1,0.3,1.0]}:
    raise ValueError('Prespecified baseline selection settings changed.')
print('Will train missing folds for:',RUN_VARIANTS)

Will train missing folds for: ['V0', 'V1', 'V2', 'V3']


## 1. Restore the frozen training data and 25 outer folds

Check the data SHA, job/group IDs, disjoint groups, fold signatures, and exactly one validation appearance per row per repeat. Save inner-fold row assignments/signatures for auditing; do not regenerate outer folds. Notebook 02 did not save historical inner assignments. V0 reproduction uses the current environment and original seed rule; all four new variants use the same inner procedure.


In [2]:
train_path=ROOT/'data/processed/d1_train.csv'
assignment_path=ROOT/'data/processed/model_cv_assignments.csv'
reference_path=ROOT/'reports/tables/model_logreg_a2_fold_scores.csv'
train=pd.read_csv(train_path)
if not train.job_id.is_unique:raise ValueError('Training IDs must be unique.')
plan02=json.loads((ROOT/'reports/model_comparison_revised/lightgbm_a2_2k_plan.json').read_text())
if sha(train_path)!=plan02['training_sha256']:raise ValueError('Frozen training data changed.')
X=build_inputs(train.copy())
X['location']=train.location
Y=train.fraudulent.to_numpy()
GROUPS=train.group_id.to_numpy()
assignments=pd.read_csv(assignment_path)
positions=pd.Series(np.arange(len(train)),index=train.job_id)
folds=[]
for (repeat,fold),rows in assignments.groupby(['repeat','fold'],sort=True):
    if not rows.job_id.is_unique or not rows.job_id.isin(positions.index).all():raise ValueError('Invalid saved IDs.')
    val=np.sort(positions.loc[rows.job_id].to_numpy())
    if not np.array_equal(rows.set_index('job_id').loc[train.iloc[val].job_id,'group_id'],GROUPS[val]):
        raise ValueError('Saved group IDs differ.')
    fit=np.setdiff1d(np.arange(len(train)),val)
    folds.append((int(repeat),int(fold),fit,val))
folds=validate_folds(folds,Y,GROUPS)
if len(folds)!=25 or {r for r,*_ in folds}!=set(range(5)):raise ValueError('Require the original 5x5 plan.')
expected={(r,f):fold_signature(Y,GROUPS,fit,val) for r,f,fit,val in folds}
reference=pd.read_csv(reference_path)
def validate_scores(frame):
    if frame.duplicated(['repeat','fold']).any():raise ValueError('Duplicate folds.')
    actual={(int(r.repeat),int(r.fold)):r.fold_signature for r in frame.itertuples()}
    if actual!=expected:raise ValueError('Fold identities differ.')
validate_scores(reference)
REFERENCE_PR_AUC=float(reference.pr_auc.mean())
print(f'{len(train):,} train rows, {Y.sum()} fraud, {len(folds)} fixed outer folds.')
print(f'Historical V0 mean PR-AUC: {REFERENCE_PR_AUC:.6f}; no reserved test data read.')
display(reference.groupby('repeat')[['train_rows','val_rows','val_fraud']].agg(['min','max']))

14,439 train rows, 693 fraud, 25 fixed outer folds.
Historical V0 mean PR-AUC: 0.604871; no reserved test data read.


train_rows        val_rows       val_fraud     
              min    max      min   max       min  max
repeat                                                
0           11550  11552     2887  2889       138  139
1           11550  11552     2887  2889       138  139
2           11550  11552     2887  2889       138  139
3           11550  11552     2887  2889       138  139
4           11550  11552     2887  2889       138  139

## 2. Resumable training and diagnostics

Each variant starts from an unfitted pipeline. Save inner candidate scores, selected settings/cutoffs, outer predictions, and primary/auxiliary metrics for each fold. Counts, rare-location backoff, and deduplication never read validation rows. Record standardised geographic coefficients and outer-validation row contributions.

**Coefficient interpretation:** frequency encoding has global coefficients for numeric features, not separate coefficients for individual cities. Aggregated `abs_count_contribution` is the absolute standardised count multiplied by its model coefficient. It describes concentration, not a probability contribution, causal effect, or full-model SHAP explanation.


In [3]:
def geography_diagnostics(fitted,variant,val,repeat,fold):
    if variant=='V0':return pd.DataFrame(),pd.DataFrame()
    features=fitted.named_steps['features']
    geo=features.named_transformers_['geography']
    encoder=geo.named_steps['count']
    raw=encoder.transform(X.iloc[val][['location','group_id']])
    scaled=geo.named_steps['scale'].transform(raw)
    coefficients=fitted.named_steps['model'].coef_[0,features.output_indices_['geography']]
    names=encoder.get_feature_names_out()
    coef=pd.DataFrame({'variant':variant,'repeat':repeat,'fold':fold,'feature':names,'coefficient':coefficients})
    detail=encoder.describe(X.iloc[val][['location','group_id']]).reset_index(drop=True)
    detail['job_id']=train.iloc[val].job_id.to_numpy()
    detail['group_id']=GROUPS[val]
    detail['variant'],detail['repeat'],detail['fold']=variant,repeat,fold
    for index,level in enumerate(encoder.levels):
        start=index*3
        detail[f'geo_{level}_abs_count_contribution']=np.abs(scaled[:,start]*coefficients[start])
        detail[f'geo_{level}_signed_count_contribution']=scaled[:,start]*coefficients[start]
        detail[f'geo_{level}_abs_geo_contribution']=np.abs(scaled[:,start:start+3]*coefficients[start:start+3]).sum(axis=1)
    return coef,detail

def run_variant(variant):
    if variant!='V0':
        if 'V0' not in RESULTS:raise ValueError('Complete/reload V0 first.')
        if abs(RESULTS['V0'].pr_auc.mean()-REFERENCE_PR_AUC)>BASELINE_TOLERANCE:
            raise ValueError('V0 reproduction differs by >0.005. Stop and investigate; do not retune folds.')
    notebook=json.loads((ROOT/'notebooks/09_geography_ablation.ipynb').read_text())
    runner=next(''.join(c['source']) for c in notebook['cells'] if c['id']=='geo-runner')
    plan={'variant':variant,'training_sha256':sha(train_path),'assignments_sha256':sha(assignment_path),
          'historical_baseline_sha256':sha(reference_path),'outer_signatures':{f'{r}:{f}':s for (r,f),s in expected.items()},
          'source_sha256':{p:sha(ROOT/'src'/p) for p in ['features.py','evaluation.py','geography.py']},
          'runner_sha256':digest(runner),'grid':PARAM_GRID,'seed':SEED,'inner_splits':INNER_SPLITS,
          'threads':THREADS,'minimum_geo_count':5,'baseline_tolerance':BASELINE_TOLERANCE,
          'estimator':repr(geographic_pipeline(variant)),
          'versions':{p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','joblib']}}
    directory=PROCESSED_DIR/variant
    plan_path=REPORT_DIR/f'{variant}_plan.json'
    if plan_path.exists() and json.loads(plan_path.read_text())!=plan:
        raise ValueError(f'{variant} cached plan differs; do not overwrite. Choose new output folders.')
    if variant in RUN_VARIANTS:
        directory.mkdir(parents=True,exist_ok=True);REPORT_DIR.mkdir(parents=True,exist_ok=True)
        write_json(plan_path,plan)
    score_rows=[];prediction_batches=[];inner_rows=[];coefficient_batches=[]
    with tempfile.TemporaryDirectory(prefix='bt4012_geo_') as cache:
        memory=Memory(cache,verbose=0)
        pipe=geographic_pipeline(variant,memory=memory,seed=SEED)
        for repeat,fold,fit,val in folds:
            marker=directory/f'r{repeat}_f{fold}.json'
            pred_path=directory/f'r{repeat}_f{fold}_predictions.csv'
            detail_path=directory/f'r{repeat}_f{fold}_geography.csv'
            if marker.exists():
                record=json.loads(marker.read_text())
                if not plan_path.exists() or record['plan_sha256']!=digest(plan) or record['score']['fold_signature']!=expected[(repeat,fold)]:
                    raise ValueError('Checkpoint configuration/folds differ.')
                if sha(pred_path)!=record['predictions_sha256']:raise ValueError('Prediction checkpoint changed.')
                if variant!='V0' and sha(detail_path)!=record['geography_sha256']:raise ValueError('Geo diagnostics changed.')
                batch=pd.read_csv(pred_path)
                if batch.job_id.tolist()!=train.iloc[val].job_id.tolist():raise ValueError('Prediction row order differs.')
                measured={**score_summary(Y[val],batch.probability.to_numpy(),record['score']['threshold']),
                          **auxiliary_metrics(Y[val],batch.probability.to_numpy())}
                if any(not np.isclose(measured[m],record['score'][m]) for m in measured):
                    raise ValueError('Checkpoint metrics do not reproduce saved predictions.')
                row,inner,coefficients=record['score'],record['inner'],record['coefficients']
                print(f'{variant}: reused repeat {repeat+1}, fold {fold+1}',flush=True)
            elif variant in RUN_VARIANTS:
                started=time.perf_counter()
                with threadpool_limits(limits=THREADS):
                    selection=select_settings(lambda:pipe,X.iloc[fit],Y[fit],GROUPS[fit],PARAM_GRID,
                        inner_splits=INNER_SPLITS,seed=SEED+repeat*1000+fold)
                    fitted=clone(pipe).set_params(**selection.params).fit(X.iloc[fit],Y[fit])
                    probabilities=fitted.predict_proba(X.iloc[val])[:,1]
                row={**score_summary(Y[val],probabilities,selection.threshold),**auxiliary_metrics(Y[val],probabilities)}
                row.update(variant=variant,repeat=repeat,fold=fold,train_rows=len(fit),val_rows=len(val),
                           val_fraud=int(Y[val].sum()),selected_params=selection.params,
                           selection_pr_auc=float(selection.candidates.mean_inner_pr_auc.max()),fold_signature=expected[(repeat,fold)])
                # JSON roundtrip normalises NumPy scalar types, preserving cutoffs at 15 significant decimals.
                row=json.loads(pd.Series(row).to_json(double_precision=15))
                inner=json.loads(selection.candidates.assign(variant=variant,repeat=repeat,fold=fold).to_json(orient='records',double_precision=15))
                coef,detail=geography_diagnostics(fitted,variant,val,repeat,fold)
                coefficients=json.loads(coef.to_json(orient='records',double_precision=15))
                batch=pd.DataFrame({'repeat':repeat,'fold':fold,'job_id':train.iloc[val].job_id.to_numpy(),
                                    'probability':probabilities,'threshold':selection.threshold})
                write_csv(pred_path,batch)
                if variant!='V0':write_csv(detail_path,detail)
                # Also save the exact inner row assignments/signatures used for this outer fit.
                from src.evaluation import make_folds
                inner_plan=[]
                for _,inner_fold,inner_fit,inner_val in make_folds(GROUPS[fit],Y[fit],n_splits=INNER_SPLITS,n_repeats=1,
                                                                 seed=SEED+repeat*1000+fold):
                    inner_plan.append({'fold':inner_fold,'fit_job_ids':train.iloc[fit[inner_fit]].job_id.tolist(),
                                       'validation_job_ids':train.iloc[fit[inner_val]].job_id.tolist(),
                                       'signature':fold_signature(Y[fit],GROUPS[fit],inner_fit,inner_val)})
                write_json(marker,{'plan_sha256':digest(plan),'score':row,'inner':inner,'inner_folds':inner_plan,
                                   'coefficients':coefficients,'predictions_sha256':sha(pred_path),
                                   'geography_sha256':sha(detail_path) if variant!='V0' else None})
                memory.clear(warn=False)
                print(f'{variant} repeat {repeat+1} fold {fold+1}: PR-AUC {row["pr_auc"]:.3f}; '
                      f'C={selection.params["model__C"]}; {time.perf_counter()-started:.1f}s',flush=True)
            else:continue
            score_rows.append(row);prediction_batches.append(batch);inner_rows.extend(inner)
            if coefficients:coefficient_batches.append(pd.DataFrame(coefficients))
        if len(score_rows)!=len(folds):
            print(f'{variant}: pending, {len(score_rows)}/25 completed. Not included in complete comparisons.')
            return
    frame=pd.DataFrame(score_rows);validate_scores(frame)
    RESULTS[variant]=frame
    export=frame.copy();export['selected_params']=export.selected_params.map(lambda p:json.dumps(p,sort_keys=True))
    write_csv(REPORT_DIR/f'{variant}_fold_scores.csv',export)
    write_csv(REPORT_DIR/f'{variant}_inner_settings.csv',pd.DataFrame(inner_rows))
    write_csv(PROCESSED_DIR/f'{variant}_predictions.csv',pd.concat(prediction_batches,ignore_index=True))
    if coefficient_batches:write_csv(REPORT_DIR/f'{variant}_geo_coefficients.csv',pd.concat(coefficient_batches,ignore_index=True))
    write_json(REPORT_DIR/f'{variant}_complete.json',{'folds':len(frame),'scores_sha256':sha(REPORT_DIR/f'{variant}_fold_scores.csv')})
    if variant=='V0':
        gap=float(frame.pr_auc.mean()-REFERENCE_PR_AUC)
        reproduction=pd.DataFrame([{'historical_mean_pr_auc':REFERENCE_PR_AUC,'reproduced_mean_pr_auc':frame.pr_auc.mean(),
                                    'difference':gap,'within_prespecified_tolerance':abs(gap)<=BASELINE_TOLERANCE}])
        write_csv(REPORT_DIR/'V0_reproduction.csv',reproduction);display(reproduction)
        if abs(gap)>BASELINE_TOLERANCE:raise ValueError('Baseline reproduction failed. Stop rather than change the experimental rules.')
    print(f'{variant} complete: mean PR-AUC={frame.pr_auc.mean():.6f}, SD={frame.pr_auc.std(ddof=1):.6f}',flush=True)

## V0 — Reproduce the A2 baseline


In [4]:
run_variant('V0')

V0 repeat 1 fold 1: PR-AUC 0.753; C=1.0; 29.4s


V0 repeat 1 fold 2: PR-AUC 0.546; C=1.0; 28.0s


V0 repeat 1 fold 3: PR-AUC 0.638; C=1.0; 29.6s


V0 repeat 1 fold 4: PR-AUC 0.628; C=1.0; 27.7s


V0 repeat 1 fold 5: PR-AUC 0.417; C=1.0; 28.4s


V0 repeat 2 fold 1: PR-AUC 0.497; C=1.0; 28.3s


... (17 outputs left out)

V0 repeat 5 fold 4: PR-AUC 0.689; C=1.0; 29.4s


V0 repeat 5 fold 5: PR-AUC 0.623; C=1.0; 36.4s


,historical_mean_pr_auc,reproduced_mean_pr_auc,difference,within_prespecified_tolerance
0,0.604871,0.604871,1.110223e-16,True


V0 complete: mean PR-AUC=0.604871, SD=0.093536


## V1 — Add state-level counts


In [5]:
run_variant('V1')

V1 repeat 1 fold 1: PR-AUC 0.755; C=1.0; 35.7s


V1 repeat 1 fold 2: PR-AUC 0.550; C=1.0; 31.6s


V1 repeat 1 fold 3: PR-AUC 0.638; C=1.0; 34.0s


V1 repeat 1 fold 4: PR-AUC 0.627; C=1.0; 30.3s


V1 repeat 1 fold 5: PR-AUC 0.416; C=1.0; 32.3s


V1 repeat 2 fold 1: PR-AUC 0.497; C=1.0; 28.9s


... (16 outputs left out)

V1 repeat 5 fold 3: PR-AUC 0.593; C=1.0; 29.1s


V1 repeat 5 fold 4: PR-AUC 0.692; C=1.0; 28.2s


V1 repeat 5 fold 5: PR-AUC 0.626; C=1.0; 28.5s


V1 complete: mean PR-AUC=0.605060, SD=0.094206


## V2 — Add state and city counts


In [6]:
run_variant('V2')

V2 repeat 1 fold 1: PR-AUC 0.764; C=1.0; 29.9s


V2 repeat 1 fold 2: PR-AUC 0.565; C=1.0; 29.3s


V2 repeat 1 fold 3: PR-AUC 0.631; C=1.0; 29.3s


V2 repeat 1 fold 4: PR-AUC 0.631; C=1.0; 28.2s


V2 repeat 1 fold 5: PR-AUC 0.426; C=1.0; 28.9s


V2 repeat 2 fold 1: PR-AUC 0.496; C=1.0; 30.3s


... (16 outputs left out)

V2 repeat 5 fold 3: PR-AUC 0.605; C=1.0; 28.8s


V2 repeat 5 fold 4: PR-AUC 0.689; C=1.0; 28.7s


V2 repeat 5 fold 5: PR-AUC 0.621; C=1.0; 29.5s


V2 complete: mean PR-AUC=0.610236, SD=0.091935


## V3 — Count distinct group–location pairs


In [7]:
run_variant('V3')

V3 repeat 1 fold 1: PR-AUC 0.760; C=1.0; 30.1s


V3 repeat 1 fold 2: PR-AUC 0.572; C=1.0; 29.5s


V3 repeat 1 fold 3: PR-AUC 0.623; C=1.0; 29.5s


V3 repeat 1 fold 4: PR-AUC 0.640; C=1.0; 28.3s


V3 repeat 1 fold 5: PR-AUC 0.429; C=1.0; 28.9s


V3 repeat 2 fold 1: PR-AUC 0.509; C=1.0; 29.8s


... (16 outputs left out)

V3 repeat 5 fold 3: PR-AUC 0.615; C=1.0; 29.7s


V3 repeat 5 fold 4: PR-AUC 0.680; C=1.0; 28.5s


V3 repeat 5 fold 5: PR-AUC 0.625; C=1.0; 29.7s


V3 complete: mean PR-AUC=0.613233, SD=0.089611


## 3. Main results and paired fold comparisons

Include complete variants only. Repeats do not create additional independent datasets. The Nadeau–Bengio correction allows for overlapping training sets, but remains approximate. Save all per-fold differences and score tables. Interpret country→state→city comparisons alongside fallback shares: a state/city feature may receive its support from country-level backoff.


In [8]:
missing=[v for v in NAMES if v not in RESULTS]
print('Status:', 'COMPLETE' if not missing else f'PARTIAL; pending {missing}')
summary=pd.DataFrame([{'variant':v,'configuration':NAMES[v],'pr_auc_mean':f.pr_auc.mean(),
                      'pr_auc_sd':f.pr_auc.std(ddof=1),
                      **{m:f[m].mean() for m in ['precision','recall','f1','precision_at_recall60',
                                                'precision_top2pct','recall_top2pct','precision_top5pct','recall_top5pct']}}
                      for v,f in RESULTS.items()])
if not summary.empty:
    display(summary.round(4));write_csv(REPORT_DIR/'comparison_summary.csv',summary)
paired_rows=[];differences=[]
if 'V0' in RESULTS:
    for variant in ['V1','V2','V3']:
        if variant not in RESULTS:continue
        stats=corrected_difference(RESULTS[variant],RESULTS['V0'])
        paired_rows.append({'candidate':variant,'reference':'V0',**stats})
        joined=RESULTS[variant][['repeat','fold','pr_auc','fold_signature']].merge(
            RESULTS['V0'][['repeat','fold','pr_auc']],on=['repeat','fold'],suffixes=('_candidate','_V0'),validate='one_to_one')
        joined['candidate']=variant;joined['difference']=joined.pr_auc_candidate-joined.pr_auc_V0
        differences.append(joined)
paired=pd.DataFrame(paired_rows)
if not paired.empty:
    display(paired.round(5));write_csv(REPORT_DIR/'paired_vs_V0.csv',paired)
    write_csv(REPORT_DIR/'per_fold_differences_vs_V0.csv',pd.concat(differences,ignore_index=True))
incremental=[]
for candidate,reference_variant in [('V1','V0'),('V2','V1'),('V3','V2')]:
    if candidate in RESULTS and reference_variant in RESULTS:
        incremental.append({'candidate':candidate,'reference':reference_variant,
                            **corrected_difference(RESULTS[candidate],RESULTS[reference_variant])})
if incremental:
    incremental=pd.DataFrame(incremental);display(incremental.round(5))
    write_csv(REPORT_DIR/'granularity_and_dedup_differences.csv',incremental)

Status: COMPLETE


,variant,configuration,pr_auc_mean,pr_auc_sd,precision,recall,f1,precision_at_recall60,precision_top2pct,recall_top2pct,precision_top5pct,recall_top5pct
0,V0,A2 30k baseline,0.6049,0.0935,0.6114,0.5351,0.5658,0.5524,0.7930,0.3261,0.5592,0.5810
1,V1,state row counts,0.6051,0.0942,0.6123,0.5379,0.5675,0.5531,0.7958,0.3273,0.5597,0.5815
2,V2,state+city row counts,0.6102,0.0919,0.6082,0.5380,0.5663,0.5576,0.7965,0.3276,0.5639,0.5859
3,V3,state+city group-location counts,0.6132,0.0896,0.6266,0.5463,0.5775,0.5731,0.8000,0.3290,0.5661,0.5882


,candidate,reference,mean_diff,sd_diff,corrected_se,low,high,pairs,mean_validation_training_ratio,t_critical
0,V1,V0,0.00019,0.00338,0.00182,-0.00357,0.00395,25,0.25,2.0639
1,V2,V0,0.00537,0.00651,0.00350,-0.00187,0.01260,25,0.25,2.0639
2,V3,V0,0.00836,0.01000,0.00538,-0.00275,0.01947,25,0.25,2.0639


,candidate,reference,mean_diff,sd_diff,corrected_se,low,high,pairs,mean_validation_training_ratio,t_critical
0,V1,V0,0.00019,0.00338,0.00182,-0.00357,0.00395,25,0.25,2.0639
1,V2,V1,0.00518,0.00685,0.00369,-0.00244,0.01279,25,0.25,2.0639
2,V3,V2,0.00300,0.00766,0.00412,-0.00551,0.01151,25,0.25,2.0639


## 4. Coefficients, location concentration, fallback shares, and V2/V3

Show the mean/SD of standardised global geographic coefficients; there are no city-specific coefficients. Aggregate absolute frequency contributions by location. Compare posting weighting with one contribution per group–location pair, using the within-pair mean. Compute concentration separately per repeat, then summarise across repeats. Analyse both requested keys and resolved fallback keys, retaining MISSING/UNSEEN. High concentration is a diagnostic clue, not proof of leakage or causation.


In [9]:
concentration=[];backoffs=[];place_details=[]
for variant in ['V1','V2','V3']:
    if variant not in RESULTS:continue
    coefficients=pd.read_csv(REPORT_DIR/f'{variant}_geo_coefficients.csv')
    print(variant,'standardised geo coefficients:')
    display(coefficients.groupby('feature').coefficient.agg(['mean','std','min','max']).round(4))
    detail=pd.concat([pd.read_csv(PROCESSED_DIR/variant/f'r{r}_f{f}_geography.csv') for r,f,*_ in folds],ignore_index=True)
    for level in (['state'] if variant=='V1' else ['state','city']):
        raw_key=f'geo_{level}_key';resolved_key=f'geo_{level}_resolved_key'
        importance=f'geo_{level}_abs_count_contribution'
        backoffs.append(detail.groupby(['repeat',f'geo_{level}_resolved_level']).size().rename('rows').reset_index()
                        .assign(variant=variant,level=level))
        for repeat,batch in detail.groupby('repeat'):
            for key_type,key_column in [('requested',raw_key),('resolved',resolved_key)]:
                for weighting in ['posting','group_location_once']:
                    data=batch if weighting=='posting' else batch.groupby(['group_id',key_column],dropna=False)[importance].mean().reset_index()
                    places=data.groupby(key_column,dropna=False)[importance].sum().sort_values(ascending=False)
                    total=float(places.sum())
                    concentration.append({'variant':variant,'level':level,'repeat':repeat,'key_type':key_type,
                        'weighting':weighting,'locations':len(places),
                        'top5_share':float(places.head(5).sum()/total) if total else 0.,
                        'top10_share':float(places.head(10).sum()/total) if total else 0.,
                        'locations_for_80pct':int(np.searchsorted(places.cumsum().to_numpy(),.8*total)+1) if total else 0})
                    top=places.head(10).rename('absolute_count_contribution').reset_index().rename(columns={key_column:'key'})
                    place_details.append(top.assign(variant=variant,level=level,repeat=repeat,key_type=key_type,weighting=weighting))
if concentration:
    concentration=pd.DataFrame(concentration)
    display(concentration.groupby(['variant','level','key_type','weighting'])[['top5_share','top10_share','locations_for_80pct']].mean().round(4))
    write_csv(REPORT_DIR/'location_concentration_by_repeat.csv',concentration)
    write_csv(REPORT_DIR/'top_locations_by_repeat.csv',pd.concat(place_details,ignore_index=True))
    backoffs=pd.concat(backoffs,ignore_index=True)
    # State and city breakdowns have different column names: combine into a common level column.
    backoffs['resolved_level']=backoffs.get('geo_state_resolved_level',pd.Series(index=backoffs.index,dtype=object)).combine_first(
                                backoffs.get('geo_city_resolved_level',pd.Series(index=backoffs.index,dtype=object)))
    backoffs['share']=backoffs.rows/backoffs.groupby(['variant','level','repeat']).rows.transform('sum')
    display(backoffs.groupby(['variant','level','resolved_level'])['share'].mean().round(4))
    write_csv(REPORT_DIR/'fallback_shares.csv',backoffs[['variant','level','repeat','resolved_level','rows','share']])

V1 standardised geo coefficients:
V2 standardised geo coefficients:
V3 standardised geo coefficients:


,mean,std,min,max
feature,,,,
geo_state_log_count,0.1735,0.0665,0.0603,0.3161
geo_state_missing,0.0908,0.0759,-0.0842,0.2162
geo_state_unseen,0.0000,0.0000,0.0000,0.0000


,mean,std,min,max
feature,,,,
geo_city_log_count,0.0200,0.0830,-0.1391,0.1649
geo_city_missing,0.4232,0.0640,0.2958,0.5510
geo_city_unseen,0.0000,0.0000,0.0000,0.0000
geo_state_log_count,0.1855,0.0848,0.0526,0.3357
geo_state_missing,-0.2175,0.0854,-0.3645,-0.0883
geo_state_unseen,0.0000,0.0000,0.0000,0.0000


,mean,std,min,max
feature,,,,
geo_city_log_count,-0.4342,0.1033,-0.6631,-0.2620
geo_city_missing,0.0938,0.0742,-0.0593,0.2533
geo_city_unseen,0.0000,0.0000,0.0000,0.0000
geo_state_log_count,0.3855,0.0943,0.2533,0.5834
geo_state_missing,-0.0794,0.0897,-0.2977,0.0759
geo_state_unseen,0.0000,0.0000,0.0000,0.0000


top5_share  ...  locations_for_80pct
variant level key_type  weighting                        ...                     
V1      state requested group_location_once      0.6450  ...                 38.0
                        posting                  0.6939  ...                 17.6
              resolved  group_location_once      0.6612  ...                 15.0
                        posting                  0.6939  ...                 10.8
V2      city  requested group_location_once      0.4441  ...                488.6
                        posting                  0.5144  ...                208.4
              resolved  group_location_once      0.5332  ...                 31.4
                        posting                  0.5517  ...                 25.8
        state requested group_location_once      0.6456  ...                 37.4
                        posting                  0.6913  ...                 18.0
              resolved  group_location_once      0.6614  ...                 14.8
                        posting                  0.6913  ...                 11.2
V3      city  requested group_location_once      0.4430  ...                503.6
                        posting                  0.5132  ...                215.4
              resolved  group_location_once      0.5643  ...                 22.6
                        posting                  0.5776  ...                 19.4
        state requested group_location_once      0.6396  ...                 43.0
                        posting                  0.6807  ...                 19.8
              resolved  group_location_once      0.6626  ...                 10.4
                        posting                  0.6807  ...                  8.8

[20 rows x 3 columns]

variant  level  resolved_level
V1       state  MISSING           0.1520
                UNSEEN            0.0067
                country           0.0543
                state             0.7870
V2       city   MISSING           0.1902
                UNSEEN            0.0046
                city              0.5134
                country           0.0489
                state             0.2428
         state  MISSING           0.1520
                UNSEEN            0.0067
                country           0.0543
                state             0.7870
V3       city   MISSING           0.1902
                UNSEEN            0.0119
                city              0.4660
                country           0.0692
                state             0.2627
         state  MISSING           0.1520
                UNSEEN            0.0150
                country           0.0754
                state             0.7575
Name: share, dtype: float64

## 5. Apply the prespecified rules; conclusion ≤150 characters

Do not change folds, cutoff rules, the C grid, or other features in response to results. An interval containing zero is a negative result under this protocol, not proof of an exactly zero effect. If only V2 passes and V3 does not, use the prespecified repeated-posting label while explicitly retaining causal uncertainty.


In [10]:
if set(RESULTS)!=set(NAMES):
    conclusion='Experiment incomplete; no gain or explanation is concluded.'
else:
    verdict=paired.set_index('candidate').copy()
    verdict['passed']=verdict.low>0
    only_v2=bool(verdict.loc['V2','passed'] and not verdict.loc['V1','passed'] and not verdict.loc['V3','passed'])
    verdict['include']=verdict.passed
    verdict['decision']=np.where(verdict.passed,'Passes prespecified gain rule','Negative result: exclude')
    if only_v2:
        verdict.loc['V2',['include','decision']]=[False,'Reposting-driven under preset rule: exclude']
        conclusion='Observed: only V2 passed; V3 did not. Exclude V2 as repetition-driven under the preset rule. Cause remains unverified.'
    else:
        passed=[v for v in ['V1','V2','V3'] if verdict.loc[v,'include']]
        observed=', '.join(passed)+' passed the corrected CI rule' if passed else 'none of V1–V3 passed'
        conclusion=f'Observed: {observed}; others excluded. Unverified: employer, sampling, or reposting effects.'
    display(verdict[['mean_diff','sd_diff','low','high','include','decision']])
    write_csv(REPORT_DIR/'decisions.csv',verdict.reset_index())
assert len(conclusion)<=150
print(conclusion)
if set(RESULTS)==set(NAMES):(REPORT_DIR/'conclusion.txt').write_text(conclusion+'\n')
print('Test set was never opened.')

Observed: none of V1–V3 passed; others excluded. Unverified: employer, sampling, or reposting effects.
Test set was never opened.


,mean_diff,sd_diff,low,high,include,decision
candidate,,,,,,
V1,0.000190,0.003384,-0.003571,0.003951,False,Negative result: exclude
V2,0.005366,0.006507,-0.001867,0.012598,False,Negative result: exclude
V3,0.008363,0.009997,-0.002748,0.019473,False,Negative result: exclude
